In [ ]:
#library imports
import pandas as pd
import numpy as np

In [ ]:
#Import data sets
#SUE
ibes_actuals = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/ibes_actuals.parquet")
ibes_summary = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/ibes_summary.parquet")
ibes_identifiers = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/ibes_identifiers.parquet")
#CAR
crsp_dsf = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/crsp_dsf.parquet")
ff_factors = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/ff_factors_daily.parquet")
#Controls
ccm_link = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/ccm_link.parquet")
compustat_fundq = pd.read_parquet("/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/compustat_fundq.parquet")

In [ ]:
#Create SUE -> unadjusted ibes actuals & unadjusted ibes summary(forcasts)
# (Actual-Predicted)/rolling 8 quarters std, use most recent forcast possible but strictly before
# actuals_col_names: oftic, anndats, value. summary_col_names: oftic, statpers, fpedats, fpi, meanest

def create_sue(summary, actuals):
    '''return dataframe with SUE based on given summary & actuals'''
    summary['statpers_dt'] = pd.to_datetime(summary['statpers'], format='%Y-%m-%d')
    summary['fpedats_dt'] = pd.to_datetime(summary['fpedats'], format='%Y-%m-%d')
    actuals['pends_dt'] = pd.to_datetime(actuals['pends'], format='%Y-%m-%d')
    actuals['anndats_dt'] = pd.to_datetime(actuals['anndats'], format='%Y-%m-%d')
    
    merged = summary.merge(actuals, how='inner', left_on=['ticker','fpedats_dt'], right_on=['ticker','pends_dt'])

    merged = merged[(merged['statpers_dt'] <= merged['anndats_dt'] - pd.Timedelta(days=1)) & (merged['fpi'] == '6')].sort_values('statpers').groupby(['ticker','anndats_dt']).last().reset_index()
    merged['numerator'] = merged['value'] - merged['meanest']

    merged = merged.sort_values(['ticker','anndats'])

    merged['sigma'] = merged.groupby('ticker')['numerator'].rolling(8, min_periods=4).std().shift(1).reset_index(level=0, drop=True)

    merged['SUE'] = merged['numerator']/merged['sigma']

    return merged[['ticker', 'anndats_dt', 'fpedats_dt', 'value', 'meanest', 'numerator', 'sigma', 'SUE']]

sue_df = create_sue(ibes_summary, ibes_actuals)
sue_df.to_parquet('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/interim/sue.parquet')

In [ ]:
#Car
#Abnormal Return = stock_return - (b1(market_return-ff) + b2(SMB) + b3(hml) + b4(rmw))
#betas made in estimation window, used in event window
events = sue_df[['ticker','anndats_dt']]
crsp_dsf['date_dt'] = pd.to_datetime(crsp_dsf['date'],format='%Y-%m-%d')
ff_factors['date_dt'] =  pd.to_datetime(ff_factors['date'],format='%Y-%m-%d')


def create_car(permino, anndats_dt, crsp_dsf, ff_factors, est_window=(-300,-90),event_window=(2,60)):
    firm = crsp_dsf.get(permino)
    if firm is None:
        return None
    firm = firm.sort_values('date_dt').reset_index(drop=True)

    after_event = firm[firm['date_dt'] >= anndats_dt]
    if after_event.empty:
        return None
    event_idx = after_event.index[0]
    firm['rel_day'] = firm.index - event_idx

    merged = firm.merge(ff_factors, how='inner', on='date_dt')
    merged['extra_return'] = merged['ret'] - merged['rf']

    est = merged[merged['rel_day'].between(*est_window)]
    est = est.dropna(subset=['extra_return','mktrf','smb','hml','rmw','cma','umd'])
    if len(est) < 30:
        return None
    factor_cols = ['mktrf','smb','hml','rmw','cma','umd']
    
    X = np.column_stack([np.ones(len(est)), est[factor_cols].to_numpy(dtype=float)])
    Y = est['extra_return'].to_numpy(dtype=float)
    betas, *_ = np.linalg.lstsq(X, Y, rcond=None)
    
    evt = merged[merged['rel_day'].between(*event_window)]
    if evt.empty:
        return None
    Xe = np.column_stack([np.ones(len(evt)), evt[factor_cols].to_numpy(dtype=float)])
    expected = Xe @ betas
    abnormal = evt['extra_return'].to_numpy(dtype=float) - expected
    return abnormal.sum()  

# IBES side: ticker → 8-digit cusip
ibes_link = ibes_identifiers[['ticker', 'cusip']].dropna().drop_duplicates()
ibes_link['cusip8'] = ibes_link['cusip'].str.strip().str[:8]

# CRSP side: 8-digit ncusip → permno
crsp_link = crsp_dsf[['permno', 'ncusip']].dropna().drop_duplicates()
crsp_link['cusip8'] = crsp_link['ncusip'].str.strip().str[:8]

# Bridge on the 8-digit cusip
ticker_to_permno = ibes_link.merge(crsp_link[['cusip8', 'permno']], on='cusip8', how='inner')[['ticker', 'permno']].drop_duplicates()

events = sue_df[['ticker', 'anndats_dt']].drop_duplicates()
events = events.merge(ticker_to_permno, on='ticker', how='inner')


for col in ['mktrf','smb','hml','rmw','cma','umd','rf']:
    ff_factors[col] = pd.to_numeric(ff_factors[col], errors='coerce')

crsp_dsf['ret'] = pd.to_numeric(crsp_dsf['ret'], errors='coerce')

crsp_by_permno = dict(tuple(crsp_dsf.groupby('permno')))

cars = []
for _, event in events.iterrows():
    car = create_car(event['permno'], event['anndats_dt'], crsp_by_permno, ff_factors)
    cars.append({'permno': event['permno'], 'ticker': event['ticker'], 'anndats_dt': event['anndats_dt'], 'CAR_2_60': car})

cars_df = pd.DataFrame(cars)

In [ ]:
cars_df.to_parquet('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/interim/cars.parquet')

In [ ]:
#Controls
#crsp_dsf, compustat_fundq, ccm_link

def add_controls(events, compustat, crsp_dsf, ccm):
    #MARKET CAP
    mktcap = crsp_dsf[['permno', 'date_dt', 'prc', 'shrout']].copy()
    mktcap['prc'] = pd.to_numeric(mktcap['prc'], errors='coerce')
    mktcap['shrout'] = pd.to_numeric(mktcap['shrout'], errors='coerce')
    mktcap['mktcap'] = mktcap['prc'].abs() * mktcap['shrout']
    mktcap = mktcap.dropna(subset=['mktcap']).sort_values('date_dt')

    events = events.sort_values('anndats_dt')
    events = pd.merge_asof(
        events, mktcap[['permno', 'date_dt', 'mktcap']],
        left_on='anndats_dt', right_on='date_dt',
        by='permno', direction='backward'
    )
    events['log_mktcap'] = np.log(events['mktcap'])

    # LINK permno to gvkey (date-aware)
    link = ccm[['permno', 'gvkey', 'linkdt', 'linkenddt']].copy()
    link['linkdt'] = pd.to_datetime(link['linkdt'])
    link['linkenddt'] = pd.to_datetime(link['linkenddt']).fillna(pd.Timestamp('2099-12-31'))
    events = events.merge(link, on='permno', how='left')
    valid = (events['anndats_dt'] >= events['linkdt']) & (events['anndats_dt'] <= events['linkenddt'])
    events = events[valid].drop(columns=['linkdt', 'linkenddt'])

    #BOOK-TO-MARKET and LEVERAGE
    fund = compustat[['gvkey', 'rdq', 'atq', 'ltq']].copy()
    fund['rdq'] = pd.to_datetime(fund['rdq'])
    fund['atq'] = pd.to_numeric(fund['atq'], errors='coerce')
    fund['ltq'] = pd.to_numeric(fund['ltq'], errors='coerce')
    fund = fund.dropna(subset=['rdq', 'atq', 'ltq'])
    fund['book_equity'] = fund['atq'] - fund['ltq']
    fund['leverage'] = fund['ltq'] / fund['atq']
    fund = fund.sort_values('rdq')

    events = events.sort_values('anndats_dt')
    events = pd.merge_asof(
        events, fund[['gvkey', 'rdq', 'book_equity', 'leverage']],
        left_on='anndats_dt', right_on='rdq',
        by='gvkey', direction='backward'
    )
    events['book_to_market'] = events['book_equity'] / events['mktcap']

    #EARNINGS VOLATILITY
    evol = compustat[['gvkey', 'rdq', 'epspxq']].copy()
    evol['rdq'] = pd.to_datetime(evol['rdq'])
    evol['epspxq'] = pd.to_numeric(evol['epspxq'], errors='coerce')
    evol = evol.dropna(subset=['rdq', 'epspxq']).sort_values(['gvkey', 'rdq'])
    evol['earnings_vol'] = (
        evol.groupby('gvkey')['epspxq']
            .rolling(8, min_periods=4).std()
            .reset_index(level=0, drop=True)
    )
    evol = evol.dropna(subset=['earnings_vol']).sort_values('rdq')

    events = events.sort_values('anndats_dt')
    events = pd.merge_asof(
        events, evol[['gvkey', 'rdq', 'earnings_vol']],
        left_on='anndats_dt', right_on='rdq',
        by='gvkey', direction='backward', suffixes=('', '_evol')
    )

    #MOMENTUM
    mom = crsp_dsf[['permno', 'date_dt', 'ret']].copy()
    mom['ret'] = pd.to_numeric(mom['ret'], errors='coerce')
    mom = mom.dropna(subset=['ret']).sort_values(['permno', 'date_dt'])
    mom['logret'] = np.log1p(mom['ret'])
    mom['cumlog'] = mom.groupby('permno')['logret'].cumsum()
    mom = mom[['permno', 'date_dt', 'cumlog']].sort_values('date_dt')

    events['anchor_near'] = events['anndats_dt'] - pd.Timedelta(days=30)
    events['anchor_far'] = events['anndats_dt'] - pd.Timedelta(days=180)

    near = mom.rename(columns={'cumlog': 'cumlog_near', 'date_dt': 'd_near'})
    events = events.sort_values('anchor_near')
    events = pd.merge_asof(events, near, left_on='anchor_near', right_on='d_near',
                           by='permno', direction='backward')

    far = mom.rename(columns={'cumlog': 'cumlog_far', 'date_dt': 'd_far'})
    events = events.sort_values('anchor_far')
    events = pd.merge_asof(events, far, left_on='anchor_far', right_on='d_far',
                           by='permno', direction='backward')

    events['momentum'] = np.expm1(events['cumlog_near'] - events['cumlog_far'])

    #clean up
    events = events.drop(columns=['anchor_near', 'anchor_far', 'd_near', 'd_far',
                                  'cumlog_near', 'cumlog_far'])
    return events.sort_values(['permno', 'anndats_dt']).reset_index(drop=True)


#---
ibes_link = ibes_identifiers[['ticker', 'cusip']].dropna().drop_duplicates()
ibes_link['cusip8'] = ibes_link['cusip'].str.strip().str[:8]

# CRSP 8-digit ncusip -> permno
crsp_link = crsp_dsf[['permno', 'ncusip']].dropna().drop_duplicates()
crsp_link['cusip8'] = crsp_link['ncusip'].str.strip().str[:8]

# bridge
ticker_to_permno = ibes_link.merge(
    crsp_link[['cusip8', 'permno']], on='cusip8', how='inner'
)[['ticker', 'permno']].drop_duplicates()

# attach permno to events
events = sue_df[['ticker', 'anndats_dt']].drop_duplicates()
events = events.merge(ticker_to_permno, on='ticker', how='inner')

crsp_dsf['date_dt'] = pd.to_datetime(crsp_dsf['date'],format='%Y-%m-%d')
ff_factors['date_dt'] =  pd.to_datetime(ff_factors['date'],format='%Y-%m-%d')

ccm_link = ccm_link.rename(columns={'lpermno': 'permno', 'lpermco': 'permco'})

controls_df = add_controls(events, compustat_fundq, crsp_dsf, ccm_link)
controls_df.to_parquet('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/interim/controls.parquet')